# Options Pricing with Machine Learning — Analysis

End-to-end analysis of a synthetic options dataset: data distributions, training dynamics, and model evaluation against classical pricing baselines.

In [ ]:
import os

# Ensure working directory is the project root regardless of where Jupyter was launched
if not os.path.exists('data'):
    os.chdir('..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

---
## Section 1 — Dataset Overview

In [ ]:
df = pd.read_csv('data/options_dataset.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.describe().round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
fig.suptitle('Feature distributions (500,000 synthetic options)', fontsize=13)

plot_cfg = [
    ('spot',   'Spot price ($)',          axes[0, 0]),
    ('strike', 'Strike price ($)',         axes[0, 1]),
    ('vol',    'Implied volatility',       axes[1, 0]),
    ('tte',    'Time to expiry (days)',    axes[1, 1]),
]
for col, xlabel, ax in plot_cfg:
    ax.hist(df[col], bins=80, color='#4C72B0', edgecolor='none', alpha=0.85)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
counts = pd.DataFrame({
    'Category': ['Call', 'Put', 'European', 'American'],
    'Count': [
        (df['type'] == 'call').sum(),
        (df['type'] == 'put').sum(),
        (df['style'] == 'european').sum(),
        (df['style'] == 'american').sum(),
    ]
})
counts['Share'] = (counts['Count'] / len(df) * 100).round(2).astype(str) + '%'
print(counts.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, (col, title) in zip(axes, [('type', 'Option type'), ('style', 'Exercise style')]):
    vals = df[col].value_counts()
    ax.bar(vals.index, vals.values, color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.5)
    ax.set_title(title)
    ax.set_ylabel('Count')
    for i, v in enumerate(vals.values):
        ax.text(i, v + 2000, f'{v:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

---
## Section 2 — Training Curves

In [ ]:
mlp_log = pd.read_csv('results/training_log_mlp.csv')
tfm_log = pd.read_csv('results/training_log_transformer.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, log, name, color in [
    (axes[0], mlp_log, 'MLP',         '#4C72B0'),
    (axes[1], tfm_log, 'Transformer', '#DD8452'),
]:
    ax.plot(log['epoch'], log['train_loss'], color=color, lw=1.8, label='Train loss')
    ax.plot(log['epoch'], log['val_loss'],   color=color, lw=1.8, ls='--', alpha=0.7, label='Val loss')
    best_ep = log.loc[log['val_loss'].idxmin()]
    ax.axvline(best_ep['epoch'], color='black', lw=0.9, ls=':', label=f'Best epoch ({int(best_ep["epoch"])})')
    ax.set_title(f'{name} — MSE loss over epochs')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE loss')
    ax.legend()

plt.suptitle('Training dynamics', fontsize=13)
plt.tight_layout()
plt.show()

print(f"MLP    — epochs: {len(mlp_log)}, best val MSE: {mlp_log['val_loss'].min():.4f}, "
      f"best epoch: {mlp_log['val_loss'].idxmin() + 1}")
print(f"Transf — epochs: {len(tfm_log)}, best val MSE: {tfm_log['val_loss'].min():.4f}, "
      f"best epoch: {tfm_log['val_loss'].idxmin() + 1}")

---
## Section 3 — Model Comparison

In [ ]:
with open('results/metrics.json') as f:
    metrics = json.load(f)

# Overall summary
rows = []
for model_name, data in metrics.items():
    m = data['overall']
    rows.append({
        'Model': model_name,
        'MAE ($)':  round(m['mae'],  3),
        'RMSE ($)': round(m['rmse'], 3),
        'R²':       round(m['r2'],   6),
        'Max Error ($)': round(m['max_error'], 3),
    })
print('=== Overall ===')
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
def breakdown_table(dimension):
    rows = []
    for model_name, data in metrics.items():
        bd = data['breakdown'].get(dimension, {})
        for bucket, m in bd.items():
            rows.append({
                'Model':  model_name,
                'Bucket': bucket,
                'MAE ($)':  round(m['mae'],  3),
                'RMSE ($)': round(m['rmse'], 3),
                'R²':       round(m['r2'],   4),
            })
    return pd.DataFrame(rows)

for dim in ('style', 'moneyness', 'expiry'):
    print(f'\n=== Breakdown by {dim} ===')
    print(breakdown_table(dim).to_string(index=False))

---
## Section 4 — Early Exercise Premium

In [ ]:
img = mpimg.imread('results/plots/early_exercise_premium.png')
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

### What this plot shows

Each point is an American option from the test set. The y-axis is the MLP's predicted price minus the Black-Scholes price for the same inputs — the **early exercise premium** implicitly learned by the network.

Key observations:

- **ITM options (red) show the largest premium**, especially at shorter expiries. These are the options most likely to benefit from early exercise: deep in-the-money puts should be exercised early when the time value of the dividend-adjusted cash exceeds the remaining optionality.
- **OTM options (blue) cluster near zero** across all expiries. Early exercise is almost never optimal for out-of-the-money options since there is no intrinsic value to capture.
- **ATM options (green) sit between the two**: there is some early exercise value but it is smaller and concentrated at longer expiries where the option spends more time near the money.
- The premium **shrinks toward zero at tte → 0** for all moneyness categories, which is mathematically correct: as expiry approaches, the binomial tree and Black-Scholes converge.

The MLP learned these patterns purely from labelled prices generated by the binomial tree — it was never explicitly told about early exercise optimality conditions.

---
## Section 5 — Vol Smile

In [ ]:
img = mpimg.imread('results/plots/vol_smile.png')
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

### Where and why MLP errors concentrate

The x-axis is moneyness (K/S): values below 1.0 are in-the-money calls or out-of-the-money puts; values above 1.0 are out-of-the-money calls or in-the-money puts.

**The baseline has essentially zero error** at every moneyness level because it re-runs the exact same formulas used to generate the data.

**MLP error pattern:**

- **Deep OTM (K/S > 1.2) has the highest MAE.** OTM option prices are very small (often < $1) and highly sensitive to vol — a small absolute error represents a large relative error. The network must learn to extrapolate into low-probability regions of the payoff distribution, which is harder.
- **ATM (K/S ≈ 1.0) has intermediate error.** The pricing function is steepest in vega near the money, making it harder to fit precisely, though the absolute prices are moderate.
- **Deep ITM (K/S < 0.8) has lower MAE.** ITM options are dominated by intrinsic value (S − K), which is a simple linear function of spot and strike. The network fits this region easily.

This shape mirrors what practitioners call the **volatility smile**: the market's implied vol is higher in the tails than at the money, reflecting that tail pricing is genuinely harder — whether for models or for markets.

---
## Section 6 — Key Takeaways

### Model performance

- The **MLP achieves R²=0.9994** and MAE=$0.677 across 50,000 held-out options, on prices ranging from near-zero to ~$250. Training took 30 epochs (~10 minutes on CPU) before early stopping.
- The **Transformer achieves R²=0.9967** and MAE=$1.618. It converged in fewer epochs (16) but each epoch was ~5× slower (~66 s vs ~13 s), and final accuracy was lower. For this tabular task with only 8 input features, the attention mechanism provides no structural advantage over dense layers.

### What the MLP learned

- **Early exercise value:** American option predictions consistently exceed Black-Scholes prices for ITM options at short-to-medium expiries — exactly where early exercise is theoretically optimal. The network inferred this from labels alone.
- **Intrinsic vs time value decomposition:** ITM options (dominated by intrinsic value) are priced most accurately. OTM options (dominated by time value and vol sensitivity) are hardest, with MAE ~8% higher than ITM.
- **Expiry difficulty:** Short-dated options (tte < 30 days) have the highest MAE ($0.90). These options have high gamma — price changes nonlinearly and rapidly near expiry — making the regression surface harder to approximate.

### Architecture conclusions

For low-dimensional tabular regression with nonlinear but smooth targets, a well-regularised MLP (BatchNorm + Dropout + AdamW) outperforms a token-based Transformer. The Transformer's inductive bias — learning pairwise feature interactions via attention — adds overhead without benefit when the feature count is small and interactions are already captured by dense layers.

### Next steps

- **Physics-informed loss:** Add a penalty for put-call parity violations and monotonicity constraints (price must increase with vol, decrease with strike for calls).
- **Implied volatility inversion:** Train a second network to invert price → IV, which is the direction practitioners actually need.
- **Greeks estimation:** Use automatic differentiation through the trained MLP to compute delta, gamma, vega, and theta without re-running the binomial tree.